# Notebook 5: Building a Complete Chatbot

## What You'll Learn
- Build a chatbot with conversation memory from scratch
- Implement Retrieval-Augmented Generation (RAG)
- Add tool use / function calling to your chatbot
- Evaluate LLM outputs systematically
- Deploy a chatbot with Gradio

---

## The Architecture of a Modern Chatbot

A production chatbot is more than just a model. It's a **system**:

```
User Message
    │
    ▼
┌─────────────────┐
│  Conversation    │ ← Maintains chat history
│  Memory Manager  │
└────────┬────────┘
         ▼
┌─────────────────┐
│  RAG: Retrieve   │ ← Search relevant documents
│  context docs    │
└────────┬────────┘
         ▼
┌─────────────────┐
│  Prompt Builder  │ ← System prompt + context + history + user query
└────────┬────────┘
         ▼
┌─────────────────┐
│  LLM Inference   │ ← Generate response
└────────┬────────┘
         ▼
┌─────────────────┐
│  Tool Router     │ ← Execute tools if needed, then re-query
└────────┬────────┘
         ▼
    Response
```

In [ ]:
# Install dependencies
# !pip install torch transformers accelerate gradio sentence-transformers faiss-cpu

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import re
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass, field

torch.manual_seed(42)

## 1. Conversation Memory

Chatbots need to remember what was said earlier in the conversation. But LLMs have a **finite context window** — we can't just feed the entire conversation history forever.

### Memory Strategies

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Full History** | Keep all messages | Perfect recall | Hits context limit |
| **Sliding Window** | Keep last N messages | Simple, bounded | Forgets early context |
| **Summarization** | Periodically summarize old messages | Compresses well | Lossy, extra inference |
| **Hybrid** | Window + summary of older messages | Best of both | More complex |

In [ ]:
@dataclass
class Message:
    role: str       # 'system', 'user', or 'assistant'
    content: str
    
    def to_dict(self):
        return {"role": self.role, "content": self.content}
    
    def token_estimate(self):
        """Rough estimate: ~4 characters per token."""
        return len(self.content) // 4


class ConversationMemory:
    """Manages conversation history with different memory strategies."""
    
    def __init__(self, max_tokens=2048, strategy='sliding_window'):
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.summary = ""  # For summarization strategy
    
    def add_message(self, role: str, content: str):
        self.messages.append(Message(role=role, content=content))
    
    def get_messages(self, system_prompt: str = "") -> List[Dict]:
        """Get the messages to include in the prompt."""
        result = []
        
        # Always include system prompt
        if system_prompt:
            result.append({"role": "system", "content": system_prompt})
        
        if self.strategy == 'full':
            result.extend([m.to_dict() for m in self.messages])
        
        elif self.strategy == 'sliding_window':
            # Keep as many recent messages as fit in the token budget
            token_count = len(system_prompt) // 4
            selected = []
            
            for msg in reversed(self.messages):
                msg_tokens = msg.token_estimate()
                if token_count + msg_tokens > self.max_tokens:
                    break
                selected.append(msg.to_dict())
                token_count += msg_tokens
            
            result.extend(reversed(selected))
        
        elif self.strategy == 'hybrid':
            # Include summary of old messages + recent messages
            if self.summary:
                result.append({
                    "role": "system",
                    "content": f"Summary of earlier conversation: {self.summary}"
                })
            
            # Add recent messages
            recent_budget = self.max_tokens // 2
            token_count = 0
            selected = []
            
            for msg in reversed(self.messages):
                msg_tokens = msg.token_estimate()
                if token_count + msg_tokens > recent_budget:
                    break
                selected.append(msg.to_dict())
                token_count += msg_tokens
            
            result.extend(reversed(selected))
        
        return result
    
    def clear(self):
        self.messages = []
        self.summary = ""
    
    def get_stats(self):
        total_tokens = sum(m.token_estimate() for m in self.messages)
        return {
            'num_messages': len(self.messages),
            'estimated_tokens': total_tokens,
            'strategy': self.strategy,
        }

# Test conversation memory
memory = ConversationMemory(max_tokens=200, strategy='sliding_window')

memory.add_message('user', 'Hi! My name is Alice.')
memory.add_message('assistant', 'Hello Alice! How can I help you today?')
memory.add_message('user', 'I want to learn about neural networks.')
memory.add_message('assistant', 'Neural networks are computational models inspired by the brain. They consist of layers of connected nodes that process information. What specific aspect interests you?')
memory.add_message('user', 'Tell me about backpropagation.')

messages = memory.get_messages(system_prompt="You are a helpful AI tutor.")
print("Messages included in prompt:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:80]}..." if len(msg['content']) > 80 else f"  [{msg['role']}]: {msg['content']}")

print(f"\n{memory.get_stats()}")

## 2. Retrieval-Augmented Generation (RAG)

LLMs have a knowledge cutoff and can hallucinate facts. **RAG** solves this by:

1. **Indexing**: Convert documents into embeddings and store in a vector database
2. **Retrieval**: When the user asks a question, find the most relevant documents
3. **Generation**: Include retrieved documents in the prompt as context

This gives the model access to up-to-date, domain-specific knowledge without retraining.

In [ ]:
# Build a simple RAG system from scratch

class SimpleVectorStore:
    """A minimal vector store for RAG (in production, use FAISS, Pinecone, etc.)."""
    
    def __init__(self, embedding_dim=384):
        self.documents = []      # Original text chunks
        self.embeddings = []     # Embedding vectors
        self.metadata = []       # Source info
        self.embedding_dim = embedding_dim
    
    def add_document(self, text: str, metadata: dict = None):
        """Add a document chunk to the store."""
        embedding = self._embed(text)
        self.documents.append(text)
        self.embeddings.append(embedding)
        self.metadata.append(metadata or {})
    
    def _embed(self, text: str) -> np.ndarray:
        """Create a simple TF-IDF-like embedding.
        In production, use sentence-transformers or OpenAI embeddings."""
        # Simple bag-of-words embedding (for demonstration)
        words = text.lower().split()
        embedding = np.zeros(self.embedding_dim)
        for word in words:
            # Hash word to an index
            idx = hash(word) % self.embedding_dim
            embedding[idx] += 1
        # Normalize
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm
        return embedding
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Find the most relevant documents for a query."""
        query_embedding = self._embed(query)
        
        # Compute cosine similarities
        similarities = []
        for i, doc_emb in enumerate(self.embeddings):
            sim = np.dot(query_embedding, doc_emb)
            similarities.append((i, sim))
        
        # Sort by similarity (descending)
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for idx, score in similarities[:top_k]:
            results.append({
                'text': self.documents[idx],
                'score': score,
                'metadata': self.metadata[idx],
            })
        
        return results

# Create a knowledge base
store = SimpleVectorStore()

knowledge_base = [
    {"text": "The Transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. It uses self-attention mechanisms instead of recurrence.", "source": "transformers.md"},
    {"text": "GPT (Generative Pre-trained Transformer) is a series of language models by OpenAI. GPT-1 had 117M parameters, GPT-2 had 1.5B, GPT-3 had 175B, and GPT-4's size is undisclosed.", "source": "gpt.md"},
    {"text": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. It freezes the pre-trained weights and adds trainable low-rank matrices to each layer, reducing trainable parameters by 99%.", "source": "lora.md"},
    {"text": "RLHF (Reinforcement Learning from Human Feedback) is used to align language models with human preferences. It involves training a reward model on human comparisons and optimizing with PPO.", "source": "rlhf.md"},
    {"text": "Tokenization in LLMs uses Byte Pair Encoding (BPE) to split text into subword units. GPT-2 uses a vocabulary of about 50,000 tokens. Each token is typically 3-4 characters.", "source": "tokenization.md"},
    {"text": "The context window of an LLM determines how much text it can process at once. GPT-3 has 2K tokens, GPT-4 has 8K-128K tokens, Claude 3 has 200K tokens. Longer contexts enable more complex reasoning.", "source": "context.md"},
    {"text": "LLaMA is an open-source LLM family by Meta. LLaMA-2 ranges from 7B to 70B parameters. LLaMA-3 was trained on 15 trillion tokens with improved tokenizer supporting 128K vocabulary.", "source": "llama.md"},
    {"text": "Attention mechanisms compute weighted sums of value vectors, where weights are determined by query-key dot products. Scaled dot-product attention divides by sqrt(d_k) to prevent softmax saturation.", "source": "attention.md"},
    {"text": "Quantization reduces model size by using lower-precision numbers. Common approaches: INT8 (8-bit), INT4 (4-bit), GPTQ, AWQ. A 70B model in float16 needs 140GB; in INT4 it needs ~35GB.", "source": "quantization.md"},
    {"text": "RAG (Retrieval-Augmented Generation) enhances LLMs by retrieving relevant documents from a knowledge base and including them in the prompt. This reduces hallucination and enables up-to-date knowledge.", "source": "rag.md"},
]

for doc in knowledge_base:
    store.add_document(doc['text'], {'source': doc['source']})

print(f'Indexed {len(knowledge_base)} documents')

In [ ]:
# Test retrieval

queries = [
    "How does LoRA work?",
    "What is the Transformer architecture?",
    "How can I reduce model size?",
]

for query in queries:
    results = store.search(query, top_k=2)
    print(f'\nQuery: "{query}"')
    for i, r in enumerate(results):
        print(f'  Result {i+1} (score: {r["score"]:.3f}, source: {r["metadata"]["source"]})')
        print(f'    {r["text"][:100]}...')

## 3. Using Real Embedding Models for RAG

Our simple bag-of-words embedding is limited. Let's use a real sentence embedding model.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a high-quality embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small but effective

class SemanticVectorStore:
    """Vector store using real sentence embeddings."""
    
    def __init__(self, embed_model):
        self.embed_model = embed_model
        self.documents = []
        self.embeddings = None
        self.metadata = []
    
    def add_documents(self, texts: List[str], metadatas: List[dict] = None):
        """Add multiple documents at once (more efficient)."""
        self.documents.extend(texts)
        
        if metadatas:
            self.metadata.extend(metadatas)
        else:
            self.metadata.extend([{} for _ in texts])
        
        # Encode all documents
        new_embeddings = self.embed_model.encode(texts, convert_to_tensor=True)
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Semantic search using cosine similarity."""
        query_emb = self.embed_model.encode([query], convert_to_tensor=True)
        
        # Cosine similarity
        similarities = F.cosine_similarity(query_emb, self.embeddings)
        
        # Top-k results
        values, indices = similarities.topk(min(top_k, len(self.documents)))
        
        results = []
        for score, idx in zip(values, indices):
            results.append({
                'text': self.documents[idx],
                'score': score.item(),
                'metadata': self.metadata[idx],
            })
        
        return results

# Index our knowledge base
semantic_store = SemanticVectorStore(embed_model)
semantic_store.add_documents(
    [doc['text'] for doc in knowledge_base],
    [{'source': doc['source']} for doc in knowledge_base]
)

print(f'Indexed {len(knowledge_base)} documents with semantic embeddings')

In [ ]:
# Compare simple vs semantic search

query = "How can I fine-tune a model efficiently without a big GPU?"

print(f'Query: "{query}"')
print(f'\n--- Simple (bag-of-words) search ---')
for r in store.search(query, top_k=2):
    print(f'  [{r["score"]:.3f}] {r["text"][:80]}...')

print(f'\n--- Semantic search ---')
for r in semantic_store.search(query, top_k=2):
    print(f'  [{r["score"]:.3f}] {r["text"][:80]}...')

print(f'\nSemantic search understands meaning, not just word overlap.')

## 4. Tool Use / Function Calling

Modern chatbots can call external tools: calculators, web search, databases, APIs.

### How Tool Use Works
1. Define available tools with their schemas
2. The LLM decides when to use a tool and generates a structured call
3. The system executes the tool and returns the result
4. The LLM incorporates the result into its response

Let's implement this pattern.

In [ ]:
class Tool:
    """Represents a tool the chatbot can use."""
    
    def __init__(self, name: str, description: str, 
                 parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters
        self.function = function
    
    def execute(self, **kwargs):
        return self.function(**kwargs)
    
    def to_schema(self) -> str:
        return json.dumps({
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
        }, indent=2)


# Define some tools

def calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    # Only allow safe math operations
    allowed_chars = set('0123456789+-*/.() ')
    if not set(expression).issubset(allowed_chars):
        return "Error: Invalid expression. Only numbers and basic operators allowed."
    try:
        result = eval(expression)  # Safe because we validated the input
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

def search_knowledge(query: str) -> str:
    """Search the knowledge base for relevant information."""
    results = semantic_store.search(query, top_k=2)
    if not results:
        return "No relevant information found."
    return "\n\n".join([f"[{r['metadata'].get('source', 'unknown')}]: {r['text']}" for r in results])

def get_current_date() -> str:
    """Get the current date and time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Register tools
tools = [
    Tool(
        name="calculator",
        description="Evaluate a mathematical expression. Use for any arithmetic.",
        parameters={"expression": "A math expression like '2 + 3 * 4'"},
        function=calculator,
    ),
    Tool(
        name="search_knowledge",
        description="Search the knowledge base for information about LLMs, Transformers, etc.",
        parameters={"query": "The search query"},
        function=search_knowledge,
    ),
    Tool(
        name="get_current_date",
        description="Get the current date and time.",
        parameters={},
        function=get_current_date,
    ),
]

print("Available tools:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

In [ ]:
# Tool execution router

class ToolRouter:
    """Parses tool calls from model output and executes them."""
    
    def __init__(self, tools: List[Tool]):
        self.tools = {t.name: t for t in tools}
    
    def get_tool_descriptions(self) -> str:
        """Generate tool descriptions for the system prompt."""
        desc = "You have access to the following tools:\n\n"
        for tool in self.tools.values():
            desc += f"Tool: {tool.name}\n"
            desc += f"Description: {tool.description}\n"
            desc += f"Parameters: {json.dumps(tool.parameters)}\n\n"
        
        desc += """To use a tool, respond with:
<tool_call>{"name": "tool_name", "arguments": {"param": "value"}}</tool_call>

After receiving the tool result, incorporate it into your response."""
        return desc
    
    def parse_and_execute(self, response: str) -> Optional[str]:
        """Parse tool calls from model response and execute them."""
        # Look for tool call pattern
        pattern = r'<tool_call>(.*?)</tool_call>'
        matches = re.findall(pattern, response, re.DOTALL)
        
        if not matches:
            return None
        
        results = []
        for match in matches:
            try:
                call = json.loads(match)
                tool_name = call.get('name')
                arguments = call.get('arguments', {})
                
                if tool_name not in self.tools:
                    results.append(f"Error: Unknown tool '{tool_name}'")
                    continue
                
                result = self.tools[tool_name].execute(**arguments)
                results.append(f"[{tool_name} result]: {result}")
                
            except json.JSONDecodeError:
                results.append("Error: Invalid tool call format")
            except Exception as e:
                results.append(f"Error executing tool: {str(e)}")
        
        return "\n".join(results)

router = ToolRouter(tools)
print("Tool descriptions for system prompt:")
print(router.get_tool_descriptions())

In [ ]:
# Test tool execution

# Simulate a model response with a tool call
simulated_response = '''I'll calculate that for you.
<tool_call>{"name": "calculator", "arguments": {"expression": "15 * 7 + 23"}}</tool_call>'''

result = router.parse_and_execute(simulated_response)
print(f'Model response: {simulated_response}')
print(f'\nTool result: {result}')

# Test knowledge search
simulated_response2 = '''Let me look that up.
<tool_call>{"name": "search_knowledge", "arguments": {"query": "LoRA fine-tuning"}}</tool_call>'''

result2 = router.parse_and_execute(simulated_response2)
print(f'\nKnowledge search result:')
print(result2)

## 5. Putting It All Together: The Complete Chatbot

Let's assemble all the components into a working chatbot.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

class Chatbot:
    """A complete chatbot with memory, RAG, and tool use."""
    
    def __init__(self, model_name="gpt2", system_prompt=None, 
                 vector_store=None, tools=None, max_memory_tokens=1024):
        
        # Load model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Components
        self.memory = ConversationMemory(
            max_tokens=max_memory_tokens, 
            strategy='sliding_window'
        )
        self.vector_store = vector_store
        self.tool_router = ToolRouter(tools) if tools else None
        
        # System prompt
        self.system_prompt = system_prompt or "You are a helpful AI assistant."
    
    def _build_prompt(self, user_message: str) -> str:
        """Build the full prompt with context, history, and tools."""
        parts = []
        
        # System prompt
        parts.append(f"System: {self.system_prompt}")
        
        # Tool descriptions
        if self.tool_router:
            parts.append(f"\n{self.tool_router.get_tool_descriptions()}")
        
        # RAG context
        if self.vector_store:
            results = self.vector_store.search(user_message, top_k=2)
            if results and results[0]['score'] > 0.3:  # Relevance threshold
                context = "\n".join([r['text'] for r in results])
                parts.append(f"\nRelevant context:\n{context}")
        
        # Conversation history
        messages = self.memory.get_messages()
        for msg in messages:
            role = msg['role'].capitalize()
            parts.append(f"\n{role}: {msg['content']}")
        
        # Current message
        parts.append(f"\nUser: {user_message}")
        parts.append("\nAssistant:")
        
        return "\n".join(parts)
    
    def chat(self, user_message: str, max_new_tokens=150, 
             temperature=0.7) -> str:
        """Send a message and get a response."""
        
        # Build prompt
        prompt = self._build_prompt(user_message)
        
        # Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt", 
                               truncation=True, max_length=1024)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        
        # Extract only the generated part
        generated = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], 
            skip_special_tokens=True
        ).strip()
        
        # Clean up: stop at next role marker
        for stop in ['\nUser:', '\nSystem:', '\nHuman:']:
            if stop in generated:
                generated = generated[:generated.index(stop)].strip()
        
        # Check for tool calls
        if self.tool_router:
            tool_result = self.tool_router.parse_and_execute(generated)
            if tool_result:
                # Add tool result and regenerate
                generated += f"\n{tool_result}"
        
        # Update memory
        self.memory.add_message('user', user_message)
        self.memory.add_message('assistant', generated)
        
        return generated
    
    def reset(self):
        """Reset conversation."""
        self.memory.clear()
        print("Conversation reset.")

# Create the chatbot
bot = Chatbot(
    model_name="gpt2",
    system_prompt="You are a helpful AI tutor specializing in machine learning and LLMs. Give concise, accurate answers.",
    vector_store=semantic_store,
    tools=tools,
)

print("Chatbot initialized!")
print(f"Model: gpt2")
print(f"Knowledge base: {len(knowledge_base)} documents")
print(f"Tools: {[t.name for t in tools]}")

In [ ]:
# Have a conversation!

conversations = [
    "What is a Transformer?",
    "How does attention work in it?",
    "What's the most efficient way to fine-tune?",
]

for user_msg in conversations:
    print(f'\nUser: {user_msg}')
    response = bot.chat(user_msg)
    print(f'Bot:  {response}')
    print('-' * 60)

print(f'\nMemory stats: {bot.memory.get_stats()}')

## 6. Evaluating LLM Outputs

How do you measure if a chatbot is good? This is one of the hardest problems in LLM development.

### Evaluation Methods

| Method | What It Measures | Automated? |
|--------|-----------------|------------|
| **Perplexity** | How well model predicts test data | Yes |
| **BLEU/ROUGE** | Overlap with reference answers | Yes |
| **LLM-as-Judge** | Quality rated by another LLM | Semi |
| **Human Evaluation** | Direct human rating | No |
| **Task-Specific** | Accuracy on benchmarks (MMLU, etc.) | Yes |

In [ ]:
# Implement evaluation metrics

def compute_perplexity(model, tokenizer, text):
    """Compute perplexity — lower is better."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss
    
    perplexity = torch.exp(loss).item()
    return perplexity

def compute_bleu(reference: str, hypothesis: str) -> float:
    """Simple BLEU score (unigram precision)."""
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    if not hyp_tokens:
        return 0.0
    
    ref_counts = Counter(ref_tokens)
    matches = 0
    
    for token in hyp_tokens:
        if ref_counts.get(token, 0) > 0:
            matches += 1
            ref_counts[token] -= 1
    
    precision = matches / len(hyp_tokens)
    
    # Brevity penalty
    bp = min(1.0, len(hyp_tokens) / max(len(ref_tokens), 1))
    
    return bp * precision

# Test evaluation
from transformers import AutoModelForCausalLM, AutoTokenizer

eval_model = AutoModelForCausalLM.from_pretrained("gpt2")
eval_tokenizer = AutoTokenizer.from_pretrained("gpt2")

test_texts = [
    "The capital of France is Paris, which is known for the Eiffel Tower.",
    "asdf jkl; qwer uiop zxcv bnm, tyui ghjk",  # Nonsense
    "Machine learning models learn patterns from data to make predictions.",
]

print("Perplexity (lower = model finds text more natural):")
for text in test_texts:
    ppl = compute_perplexity(eval_model, eval_tokenizer, text)
    print(f'  {ppl:8.1f} | "{text[:60]}..."')

print("\nBLEU Score (higher = more overlap with reference):")
ref = "The capital of France is Paris."
hypotheses = [
    "The capital of France is Paris.",       # Perfect match
    "Paris is the capital city of France.",   # Same meaning, different words
    "France is a country in Europe.",         # Related but different
    "The weather is nice today.",             # Unrelated
]

for hyp in hypotheses:
    score = compute_bleu(ref, hyp)
    print(f'  {score:.3f} | "{hyp}"')

In [ ]:
# LLM-as-Judge evaluation framework

def create_judge_prompt(question, response, criteria):
    """Create a prompt for LLM-as-Judge evaluation."""
    return f"""Rate the following response on a scale of 1-5 for each criterion.

Question: {question}

Response: {response}

Criteria:
{chr(10).join(f'- {c}' for c in criteria)}

For each criterion, provide a score (1-5) and brief justification.
"""

# Define evaluation criteria
criteria = [
    "Accuracy: Is the information correct?",
    "Helpfulness: Does it answer the question?",
    "Conciseness: Is it appropriately concise?",
    "Safety: Does it avoid harmful content?",
]

# Create evaluation prompt
question = "What is machine learning?"
response = "Machine learning is a subset of AI where computers learn from data."

judge_prompt = create_judge_prompt(question, response, criteria)
print("Judge prompt (would be sent to a capable LLM like GPT-4):")
print(judge_prompt)
print("\nIn practice, you'd send this to a stronger model to get scores.")
print("This is how companies evaluate chatbot quality at scale.")

## 7. Deploying with Gradio

Let's create a web interface for our chatbot.

In [ ]:
import gradio as gr

# Create a new chatbot instance for the UI
ui_bot = Chatbot(
    model_name="gpt2",
    system_prompt="You are a helpful AI assistant. Be concise and accurate.",
    vector_store=semantic_store,
    tools=tools,
)

def respond(message, history):
    """Gradio chat callback."""
    response = ui_bot.chat(message, max_new_tokens=150, temperature=0.7)
    return response

# Build the Gradio interface
demo = gr.ChatInterface(
    fn=respond,
    title="LLM Chatbot",
    description="A chatbot with conversation memory, RAG, and tool use. Built from scratch!",
    examples=[
        "What is the Transformer architecture?",
        "How does LoRA work?",
        "What's 42 * 17?",
    ],
    theme="soft",
)

# Launch (uncomment to run)
# demo.launch(share=False)

print("Gradio interface created!")
print("Uncomment demo.launch() to start the web interface.")
print("It will be available at http://localhost:7860")

## 8. Advanced: Document Chunking for RAG

Real RAG systems need to split long documents into chunks. Let's implement smart chunking.

In [ ]:
class DocumentChunker:
    """Split documents into chunks for RAG indexing."""
    
    def __init__(self, chunk_size=500, chunk_overlap=50):
        self.chunk_size = chunk_size  # characters per chunk
        self.chunk_overlap = chunk_overlap
    
    def chunk_by_sentences(self, text: str) -> List[str]:
        """Split text into chunks at sentence boundaries."""
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            if current_length + len(sentence) > self.chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                # Overlap: keep last sentence(s)
                overlap_length = 0
                overlap_sentences = []
                for s in reversed(current_chunk):
                    if overlap_length + len(s) > self.chunk_overlap:
                        break
                    overlap_sentences.insert(0, s)
                    overlap_length += len(s)
                current_chunk = overlap_sentences
                current_length = overlap_length
            
            current_chunk.append(sentence)
            current_length += len(sentence)
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks
    
    def chunk_by_paragraphs(self, text: str) -> List[str]:
        """Split text into chunks at paragraph boundaries."""
        paragraphs = text.split('\n\n')
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for para in paragraphs:
            if current_length + len(para) > self.chunk_size and current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = []
                current_length = 0
            
            current_chunk.append(para)
            current_length += len(para)
        
        if current_chunk:
            chunks.append('\n\n'.join(current_chunk))
        
        return chunks

# Test
long_document = """The Transformer architecture revolutionized natural language processing when it was introduced in 2017. Unlike previous approaches that used recurrent neural networks (RNNs) or convolutional neural networks (CNNs), the Transformer relies entirely on attention mechanisms. This allows it to process all tokens in a sequence simultaneously, rather than one at a time.

The key innovation is the self-attention mechanism. For each token in the input, it computes attention scores with every other token. These scores determine how much each token should influence the representation of every other token. The computation involves three learned projections: queries, keys, and values.

Multi-head attention extends this by running multiple attention computations in parallel. Each "head" can learn to attend to different types of relationships. For example, one head might learn syntactic relationships while another learns semantic ones. The outputs from all heads are concatenated and projected.

Modern large language models like GPT-4 and LLaMA use decoder-only Transformers. These use causal (masked) attention, where each token can only attend to previous tokens. This enables autoregressive generation: the model generates text one token at a time, each time conditioning on all previously generated tokens."""

chunker = DocumentChunker(chunk_size=300, chunk_overlap=50)
chunks = chunker.chunk_by_sentences(long_document)

print(f'Document length: {len(long_document)} chars')
print(f'Number of chunks: {len(chunks)}')
for i, chunk in enumerate(chunks):
    print(f'\nChunk {i+1} ({len(chunk)} chars):')
    print(f'  "{chunk[:100]}..."')

## 9. Exercises

### Exercise 1: Build a Domain-Specific Chatbot
Pick a topic (cooking, fitness, coding, etc.). Create a knowledge base of 20+ documents and build a RAG-powered chatbot for that domain. Test it with 10 questions and evaluate the quality.

In [ ]:
# EXERCISE 1: Domain-specific chatbot
# Your code here

### Exercise 2: Add More Tools
Extend the tool system with:
- A weather lookup tool (mock data is fine)
- A text summarizer tool
- A code executor tool (sandboxed!)

Create conversations that use multiple tools in sequence.

In [ ]:
# EXERCISE 2: More tools
# Your code here

### Exercise 3: Conversation Memory Evaluation
Create a test suite that verifies conversation memory works correctly:
- Can the bot remember the user's name?
- Can it reference something said 5 messages ago?
- Does the sliding window correctly truncate old messages?
- Compare full history vs. sliding window vs. hybrid strategies

In [ ]:
# EXERCISE 3: Memory evaluation
# Your code here

### Exercise 4: Swap in a Better Model
Replace GPT-2 with a better open-source model:
- `microsoft/phi-2` (2.7B, good quality)
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (1.1B, instruction-tuned)
- `mistralai/Mistral-7B-Instruct-v0.2` (7B, if you have GPU)

Compare the quality of responses.

In [ ]:
# EXERCISE 4: Better model
# Your code here

## 10. Summary: The Complete LLM Journey

### What We've Built Across 5 Notebooks

| Notebook | Topic | Key Skills |
|----------|-------|------------|
| **1** | Neural Text Generation | Bigram models, MLP language models, cross-entropy loss |
| **2** | Tokenization & Embeddings | BPE from scratch, Word2Vec, positional encodings |
| **3** | Transformer Architecture | Self-attention, multi-head attention, full GPT model |
| **4** | Training & Fine-Tuning | Pre-training, SFT, LoRA, RLHF/DPO |
| **5** | Building a Chatbot | Memory, RAG, tool use, evaluation, deployment |

### Key Takeaways

1. **LLMs are next-token predictors** — this simple objective, at scale, produces remarkable capabilities
2. **Transformers** replaced RNNs by using self-attention for parallel processing
3. **Pre-training + Fine-tuning + Alignment** is the recipe for a chatbot
4. **LoRA** makes fine-tuning accessible on consumer hardware
5. **A chatbot is a system**, not just a model — memory, retrieval, and tools are essential
6. **Evaluation is hard** — use multiple methods (automated metrics + human evaluation)

### Where to Go Next

- **Scale up**: Fine-tune LLaMA or Mistral on your own data
- **Build agents**: Use frameworks like LangChain or LlamaIndex  
- **Learn multimodal**: Vision-language models (LLaVA, GPT-4V)
- **Read the papers**: "Attention Is All You Need", "Language Models are Few-Shot Learners" (GPT-3), "LLaMA"
- **Contribute to open source**: HuggingFace, vLLM, llama.cpp